# Preprocessing

In this step, we import the necessary Python libraries:

NumPy (`numpy`): Useful for handling numerical operations, though we may not use it extensively here.

Pandas (`pandas`): Provides powerful tools for reading, organizing, and manipulating tabular data—perfect for working with our tweet dataset.

Ollama: This is the interface we use to interact with local large language models (LLMs), which will help us classify the stance of tweets related to immigration.

These libraries set the foundation for our annotation pipeline.

In [27]:
import numpy as np
import pandas as pd
import ollama

### Setting Up Ollama for Text Annotation

Before running this code, you need to have the Ollama framework installed on your machine. Ollama is a local language model hosting platform that allows you to run models like `llama3` or `mistral` on your computer without relying on cloud APIs. To get started, first visit the [Ollama website](https://ollama.com/) and follow the installation instructions for your operating system.

After installing Ollama, you must download the language model you plan to use (for example, `llama3` or `mistral`). You can do this by running commands like `ollama pull llama3` in your terminal or command prompt. This downloads the model locally, enabling fast and private inference. Bear in mind that these models will then also remain on your hard drive. They come in different sizes, so choose one that fits your hardware capabilities and needs. If you want to delete the model later, you can use the command `ollama rm llama3` (or the name of the model you downloaded and want to get rid of).

Once Ollama and the required model are installed, the Python `ollama` package can communicate with the local Ollama server to generate responses. Make sure the Ollama service is running before executing your Python scripts that call `ollama.generate()`. This setup ensures that your text annotation process runs smoothly and efficiently on your own hardware.


### Loading and Preparing the Tweet Dataset

Here we start by loading our dataset of German parliamentary tweets that have already been classified as related to immigration. The dataset is stored in a Parquet file, which is a common and efficient format for large tables.

We then filter the data to keep only the tweets that have been predicted as related to immigration (`predicted == 1`). This gives us a clean subset of relevant tweets to work with.

To make the analysis or annotation process more manageable, we divide the dataset into four equal parts. This can be useful if we want to process tweets in chunks, either for parallel processing or just to work in smaller batches.

In [28]:
import pandas as pd

# Read and filter the data
tweets = pd.read_parquet(r'C:\Users\24558\Desktop\chapter-2\data\immigration_classified_tweets_german_parl.parquet')
tweets = tweets[tweets['predicted'] == 1]

# Compute the length of each quarter
n = len(tweets) // 4

# Divide the DataFrame into four parts
tweets1 = tweets[:n]
tweets2 = tweets[n:2*n]
tweets3 = tweets[2*n:3*n]
tweets4 = tweets[3*n:]



In [29]:
tweets4

,user_username,text,created_at,tweet_id,author_id,party,source_type,year_month,predicted
872377,Halina_Waw,@Erbloggtes leider ja. deutsche bevölkerung sind die mit staatsbürgerschaft.,2013-11-18 15:31:13+00:00,402458982197252096,18866407,Linke,twitter_parliamentarian,2013-11-17 00:00:00+00:00,1
872378,Halina_Waw,"merkt mal an: nicht nur die telekommunikation der deutschen bevölkerung sollt sicher sein, sondern die aller hier lebenden menschen",2013-11-18 15:25:54+00:00,402457643165679616,18866407,Linke,twitter_parliamentarian,2013-11-17 00:00:00+00:00,1
872381,Halina_Waw,"sehr lustig. der ehemalig geheimdienstkoordinator fragt, wann das angefangen hat mit der nsa. er müsste es doch wissen bundestag",2013-11-18 14:52:37+00:00,402449270928080896,18866407,Linke,twitter_parliamentarian,2013-11-17 00:00:00+00:00,1
872407,Halina_Waw,"hat gestrichen &amp; gesäubert, nun wird repräsentiert (kurdistan kultur verein). nur mit kingstons (zwergspitz) freundschaft hapert es noch.",2013-11-15 15:32:20+00:00,401372099044196354,18866407,Linke,twitter_parliamentarian,2013-11-10 00:00:00+00:00,1
872451,Halina_Waw,zahlen gegen vorurteile. einfach mal lesen flüchtlinge,2013-11-09 08:54:19+00:00,399097607353806848,18866407,Linke,twitter_parliamentarian,2013-11-03 00:00:00+00:00,1
...,...,...,...,...,...,...,...,...,...
1060633,Katrin_Werner,Am Samstag wurde eine Frau aus Hamburg in die Abschiebehaft in Ingelheim gebracht. Ihr droht – trotz schwerer psychischer Erkrankung – die Abschiebung. Rheinland-Pfalz darf nicht länger bei Abschiebungen mitwirken!\n\nRLP RefugeesWelcome dielinke,2018-08-07 12:29:28+00:00,1026807538132688896,224630872,Linke,twitter_parliamentarian,2018-08-05 00:00:00+00:00,1
1060641,Katrin_Werner,"Statt die drängenden Probleme wie Pflegenotstand, Mietenexplosion, Armut und Integration anzugehen, hat die Regierung wieder erfolgreich das Asylrecht geschleift. Horst Seehofer ""Innenminister auf Abruf“.",2018-07-09 06:51:15+00:00,1016213172594905088,224630872,Linke,twitter_parliamentarian,2018-07-08 00:00:00+00:00,1
1060649,Katrin_Werner,"Über 68 Mio. Menschen sind auf der Flucht. Sie verlassen Familie und Freunde, ohne zu wissen, ob sie jemals zurückkehren. Wir brauchen einen Masterplan zur Bekämpfung von Fluchtursachen - nicht Geflüchteten. weltfluechtlingstag weltflüchtlingstag",2018-06-20 07:21:25+00:00,1009335395002257408,224630872,Linke,twitter_parliamentarian,2018-06-17 00:00:00+00:00,1
1060653,Katrin_Werner,"Wir LINKEN fordern: Konsequente Bekämpfung von sexistischer Gewalt, unabhängig von der Herkunft der Täter sowie ein Ende der rassistischen Instrumentalisierung von Verbrechen an Mädchen und Frauen.\n\ndielinke rlp sexismus Rassismus",2018-06-08 15:08:24+00:00,1005104260923707392,224630872,Linke,twitter_parliamentarian,2018-06-03 00:00:00+00:00,1


### Cleaning the Tweet Text

Before sending the tweets to a language model for annotation, we need to do some basic cleaning. Tweets often contain elements like URLs and hashtags that can distract the model from the actual content.

First, we define a function to remove any URLs from the tweet text. These usually start with "http" or "https" and are not relevant for understanding the tweet's stance. Then, we define another small function to remove the hashtag symbol (`#`). Instead of removing the entire hashtagged word (which could carry important meaning), we simply strip the symbol itself. Both functions are applied to the `text` column of our DataFrame, so that every tweet is cleaned in the same way before further processing.


In [30]:
# import regex library 
import re

# define function to remove urls using a regex
def remove_urls(text):
    return re.sub(r'http[s]?://\S+', '', text)

# Apply the function to the 'text' column in your DataFrame
tweets['text'] = tweets['text'].apply(remove_urls)

# define function to remove hashtags using a regex
def remove_hashtags(text):
    return re.sub(r'#', '', text)

# Apply the function to the 'text' column in your DataFrame
tweets['text'] = tweets['text'].apply(remove_hashtags)

# Creating classification Pipeline

### Defining the Function to Query the Language Model

This function is the core of our annotation process. It sends each tweet to a language model (through the Ollama framework) and returns the model's generated response.

The function takes a single row from the DataFrame, a prompt template, the model name, and a temperature setting. It first formats the prompt by inserting the tweet text into it.

We then call the `ollama.generate()` function, which sends the prompt to the model. The temperature controls how creative or consistent the model is in its output—lower values mean more predictable responses. If you want a purely deterministic (and subsequently reproducible model) you will have to set the temperature to 0 and always use the same `seed`. 

If the model successfully returns a response, we extract just the generated part. If something goes wrong (e.g., due to an API error), the function prints an error message and returns `None`, so that processing can continue without crashing.


In [4]:
def process_text(row, model, prompt, temperature):
    """
    Processes a single tweet’s text by sending it to the language model and returning the generated response.

    Parameters:
    - row: a single row from the DataFrame containing the tweet text and metadata
    - model: the name of the language model to use (e.g., 'llama3' or 'mistral')
    - prompt: a string template with a placeholder '{text}' where the tweet text will be inserted
    - temperature: controls randomness in the model’s output (0 means deterministic)

    Returns:
    - The generated response from the model as a string, or None if there was an error.
    """

    # Insert the tweet text into the prompt template
    full_prompt = prompt.format(text=row['text'])

    try:
        # Call the Ollama API to generate a response from the model
        response = ollama.generate(
            model=model,              # Specify which model to use
            prompt=full_prompt,       # The prompt with the tweet text inserted
            options={
                "temperature": temperature,  # Control randomness: 0 = deterministic
                "seed": 42,                  # Fixed seed for reproducibility
                "gpu_layers": 100,           # Number of layers to run on GPU for speed
                "use_gpu": True              # Enable GPU acceleration if available
            }
        )
        # Extract and return only the generated text response
        return response["response"]

    except Exception as e:
        # Print an error message including the tweet’s unique ID for debugging
        print(f"Error processing text {row['id']} with model {model}: {e}")
        # Return None so the caller knows this tweet failed to process
        return None

### Running Scaled Prompts Across Tweets

Below we create a function to automate the process of running multiple stance-detection prompts on a set of tweets. For each tweet, we generate responses using three types of prompts:
- **Original** prompts (e.g., asking whether the tweet supports immigration),
- **Opposite** prompts (asking whether the tweet opposes immigration),
- **Neutral** prompts (asking whether the tweet only contains neutral/factual information on immigration).

We do this for each tweet in the DataFrame and store each response in a new column so we can later compare the outputs side by side.

To speed up the annotation process, we use Python’s `ThreadPoolExecutor`, which allows us to run multiple tasks in parallel. This means that instead of processing one tweet at a time (which would be very slow when working with hundreds or thousands of tweets), we can send multiple tweets to the model at the same time. This kind of parallelism is called *multithreading*. It’s especially useful when the main bottleneck is waiting for the model to respond. By starting multiple threads, we can keep the system busy and make better use of our hardware. The number of threads used at once is controlled by the `max_workers` parameter. If your computer has a strong CPU and GPU, you can increase this number to make things even faster. If you run into memory or stability issues, lowering it can help. Overall, using threads here helps us scale up the annotation process and makes it feasible to analyze large datasets in a reasonable amount of time. 

We also use `tqdm` to display a progress bar, so we can track how the annotation is going in real time. The end result is a new DataFrame where each tweet has a set of model-generated responses, one for each type of prompt.


In [5]:
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

def generate_scaled_texts(df, model, prompts_with_values, opposite_prompts_with_values, neutral_prompts_with_values, temperature, num_workers=4):
    """
    Classifies tweets using multiple prompts—positive, opposite (negative), and neutral—and stores each model response in separate columns.

    Parameters:
    - df: DataFrame containing tweets to classify
    - model: the language model name to use (e.g., "llama3")
    - prompts_with_values: dictionary of positive prompts (keys as IDs, values as prompt strings)
    - opposite_prompts_with_values: dictionary of opposite (negative) prompts with the same keys
    - neutral_prompts_with_values: dictionary of neutral prompts with the same keys
    - temperature: controls randomness of model output (0 = deterministic)
    - num_workers: number of threads to run in parallel for faster processing

    Returns:
    - DataFrame with original tweet data plus model responses for each prompt type
    """

    results = []  # (Not used in this version but could store intermediate results)
    tasks = []    # List to hold tasks as tuples (row, model, temperature)

    # Create column names for the output DataFrame based on prompt IDs
    prompt_columns = {prompt_id: f"Prompt_{prompt_id}" for prompt_id in prompts_with_values.keys()}
    opposite_prompt_columns = {f"Opposite_{prompt_id}": f"Opposite_{prompt_id}" for prompt_id in opposite_prompts_with_values.keys()}
    neutral_prompt_columns = {f"Neutral_{prompt_id}": f"Neutral_{prompt_id}" for prompt_id in neutral_prompts_with_values.keys()}

    # Prepare tasks by iterating through each tweet (row) in the DataFrame
    for _, row in df.iterrows():
        tasks.append((row, model, temperature))

    # Dictionary to store all results keyed by unique tweet_id
    output_data = {}

    # Use ThreadPoolExecutor to run multiple tasks in parallel for speed-up
    with ThreadPoolExecutor(max_workers=num_workers) as executor:
        # tqdm shows a progress bar while processing
        for row, model, temperature in tqdm(tasks, desc="Processing Tweets", unit="tweet"):
            tweet_id = row['tweet_id']  # Unique identifier for each tweet
            
            # Initialize the dictionary entry for this tweet if not already present
            if tweet_id not in output_data:
                output_data[tweet_id] = {
                    **row.to_dict(),  # Copy all original tweet data
                    "Model": model,
                    "Temperature": temperature
                }

            # Loop through each prompt ID and prompt text in the positive prompts dictionary
            for prompt_id, prompt in prompts_with_values.items():
                # Generate the model response for the original (positive) prompt
                response = process_text(row, model, prompt, temperature)
                column_name = prompt_columns[prompt_id]
                output_data[tweet_id][column_name] = response

                # Generate the model response for the opposite (negative) prompt if available
                opposite_prompt = opposite_prompts_with_values.get(prompt_id, "")
                if opposite_prompt:
                    opposite_response = process_text(row, model, opposite_prompt, temperature)
                    opposite_column_name = opposite_prompt_columns[f"Opposite_{prompt_id}"]
                    output_data[tweet_id][opposite_column_name] = opposite_response

                # Generate the model response for the neutral prompt if available
                neutral_prompt = neutral_prompts_with_values.get(prompt_id, "")
                if neutral_prompt:
                    neutral_response = process_text(row, model, neutral_prompt, temperature)
                    neutral_column_name = neutral_prompt_columns[f"Neutral_{prompt_id}"]
                    output_data[tweet_id][neutral_column_name] = neutral_response

    # Convert the results dictionary values (one dict per tweet) into a DataFrame and return it
    return pd.DataFrame(output_data.values())


### Prompts

### Defining the Prompts and Model Parameters

Here we define the core prompts that will be sent to the language model for each tweet. These prompts are designed to ask the model whether a tweet expresses a **positive**, **negative**, or **neutral** stance on immigration-related topics (immigration, integration, or refugees). Each prompt includes a slot (`{text}`) where the tweet will be inserted. These prompts are relatively short and concise. Depending on your model's capabilities, you can want to adjust the wording or length to get the best results. Even with the model that we are using, we could expand these prompts. Be careful though: These are decoder models and they only run from left to right over any tokens they are being fed. A very long prompt might lead to the model "forgetting" what you asked it to do at the beginning of your prompt. 

We organize the prompts into three dictionaries:
- `prompts_with_values`: These are the "main" prompts that focus on identifying different levels of positive stance—regular, hidden, and slight.
- `opposite_prompts_with_values`: These ask the same questions, but from the opposite (negative) angle. This lets us check how consistent or biased the model is in its judgments.
- `neutral_prompts_with_values`: These look for neutrality, both explicit and implicit, which helps us capture tweets that avoid taking a clear stance.

Each dictionary uses a numeric key (1, 2, 3) to label different prompt variants, so we can easily keep track of which one generated each response.

We also set the model to `"llama3"`—this tells the Ollama framework which locally hosted language model we want to use. Finally, we set the `temperature` to `0.0`. A temperature of zero ensures that the model produces consistent, repeatable answers without randomness.

Most importantly, these variables (`model`, the three prompt dictionaries, and `temperature`) are exactly the values that our earlier function `generate_scaled_texts()` expects. By defining them here, we’re preparing the inputs that will be passed to that function to carry out the actual annotation work.


In [6]:
model = "llama3"  # Specify the name of the language model to use

# Dictionary of prompts asking if a tweet expresses a positive stance.
# Each key represents a different variant of the prompt.
prompts_with_values = {
    1: "Does this tweet have a positive stance immigration, integration, or refugees? Only reply with yes or no: {text}",
    2: "Does this tweet have a hidden positive stance immigration, integration, or refugees? Only reply with yes or no: {text}",
    3: "Does this tweet have a slightly positive stance immigration, integration, or refugees? Only reply with yes or no: {text}",
}

# Dictionary of opposite prompts asking if a tweet expresses a negative stance,
# using the same keys to keep the prompt variants aligned.
opposite_prompts_with_values = {
    1: "Does this tweet have a negative stance on immigration, integration, or refugees? Only reply with yes or no: {text}",
    2: "Does this tweet have a hidden negative stance on immigration, integration, or refugees? Only reply with yes or no: {text}",
    3: "Does this tweet have a slightly negative stance on immigration, integration, or refugees? Only reply with yes or no: {text}"
}

# Dictionary of neutral stance prompts, also using matching keys for variants.
neutral_prompts_with_values = {
    1: "Does this tweet have a neutral stance on immigration, integration, or refugees? Only reply with yes or no: {text}",
    2: "Does this tweet have a hidden neutral stance on immigration, integration, or refugees? Only reply with yes or no: {text}",
    3: "Does this tweet hav


### Classification Step

### Running the Annotation Process

Now that we have defined the model and prompts, we call the `generate_scaled_texts()` function to start classifying the tweets.

We pass in the following:

- `df=tweets`: This is our DataFrame containing all the tweets we want to analyze.
- `model=model`: The language model we want to use (here, `"llama3"`).
- The three sets of prompts (`prompts_with_values`, `opposite_prompts_with_values`, and `neutral_prompts_with_values`) which define the questions we ask the model.
- `temperature=temperature`: Controls the randomness of the model’s answers, set to 0 for consistency.
- `num_workers=8`: This sets how many parallel threads we use to speed up processing. Using more workers allows us to classify more tweets at the same time, taking advantage of multi-core CPUs and GPUs.

The function will return a new DataFrame called `scaled_results` that contains the original tweet data along with the model’s responses to each prompt. This makes it easy to compare the different stances the model detects.


In [7]:
scaled_results = generate_scaled_texts(
    df=tweets,  # The DataFrame containing tweets to be classified
    model=model,  # The language model we defined earlier (e.g., "llama3")
    prompts_with_values=prompts_with_values,  # Dictionary of positive stance prompts
    opposite_prompts_with_values=opposite_prompts_with_values,  # Dictionary of negative stance prompts
    neutral_prompts_with_values=neutral_prompts_with_values,  # Dictionary of neutral stance prompts
    temperature=temperature,  # Temperature setting for model output randomness (0.0 means deterministic)
    num_workers=8  # Number of threads to run in parallel, speeding up the annotation process
)


Processing Tweets:   0%|          | 0/73017 [00:04<?, ?tweet/s]


KeyboardInterrupt: 

# Batch Classification, multiple calls to API

When working with large datasets—like thousands of tweets—it’s often impractical to process each item one by one manually. Instead, we group the data into **batches**, which are smaller chunks of the full dataset. Processing data in batches allows us to send multiple requests to the language model efficiently, either sequentially or in parallel.

This approach not only saves time but also helps ensure consistency in how each text is evaluated. By making multiple calls to the model API with different prompts or models, we can speed up the entire process. This, however, relies on more computing power and memory, so it’s important to balance the number of requests with the available resources. Below you can find a code that parallelizes all of this. 


### Switching to a Different Model and Adjusting Prompts

In this step, we switch to using the `"mistral"` model instead of `"llama3"`. Different models might have slightly different strengths, so experimenting with multiple options can help improve annotation quality.

We also adjust the prompts to focus on just two levels of stance—regular and "slightly" positive, negative, or neutral—instead of three. This simplification can make the classification task clearer and reduce ambiguity in the model's responses.

As before, the temperature remains at `0.0` to ensure consistent and reproducible answers.

Just like before, these variables prepare the inputs for our annotation function, which will use this new model and prompt setup.


In [31]:
model = "mistral"  # Single model to use
prompts_with_values = {
    1: "Does this tweet have a positive stance immigration, integration, or refugees? Only reply with yes or no: {text}",
    2: "Does this tweet have a slightly positive stance immigration, integration, or refugees? Only reply with yes or no: {text}",
}
opposite_prompts_with_values = {
    1: "Does this tweet have a negative stance on immigration, integration, or refugees? Only reply with yes or no: {text}",
    2: "Does this tweet have a slightly negative stance on immigration, integration, or refugees? Only reply with yes or no: {text}"
}
neutral_prompts_with_values = {
    1: "Does this tweet have a neutral stance on immigration, integration, or refugees? Only reply with yes or no: {text}",
    2: "Does this tweet have a slightly neutral stance on immigration, integration, or refugees? Only reply with yes or no: {text}"
}
temperature = 0.0  # Single temperature value to use

### Processing a Single Tweet with the Language Model

This function `process_text` below sends one tweet’s text to the language model and returns the model’s response.

- It takes a row from the DataFrame (which contains the tweet), the model name, the prompt template, and a temperature value.
- It formats the prompt by inserting the tweet text where `{text}` appears.
- Then it calls the `ollama.generate()` function to get the model’s answer, with settings like temperature for response randomness, a fixed seed for reproducibility, and GPU usage for faster processing.
- If the call is successful, it extracts and returns only the generated response.
- If there’s an error (like a connection issue), it prints an error message including the tweet’s ID and returns `None` so the process can continue smoothly.

This function is the building block for annotating tweets one by one.


In [1]:
def process_text(row, model, prompt, temperature):
    """
    Processes a single text request and returns only the generated response for the prompt.
    """
    # Insert the tweet text into the prompt template
    full_prompt = prompt.format(text=row['text'])  
    
    try:
        # Call the Ollama model with the prepared prompt and options
        response = ollama.generate(
            model=model,                      # Specify which model to use
            prompt=full_prompt,               # Send the formatted prompt text
            options={
                "temperature": temperature,  # Control randomness of output
                "seed": 42,                  # Fix random seed for reproducibility
                "gpu_layers": 100,           # Use GPU acceleration for speed
                "use_gpu": True              # Enable GPU usage if available
            }
        )
        # Return only the text response from the model
        return response["response"]  
    
    except Exception as e:
        # Print error message if something goes wrong, including tweet id and model name
        print(f"Error processing text {row['id']} with model {model}: {e}")
        # Return None so the program can continue without crashing
        return None  


### Batch Processing with Parallel Calls to the Model

This updated version of the `generate_scaled_texts` function processes the tweets **in batches** rather than one by one.

**Key differences from the previous version:**

- **Batching:** Instead of iterating over every tweet individually, we split the DataFrame into smaller chunks called batches (default size is 16 tweets per batch). Processing in batches is often more efficient, especially when working with large datasets or APIs that perform better with grouped requests.
  
- **Parallel execution with ThreadPoolExecutor:** We still use multithreading to run multiple tasks in parallel, but now each task corresponds to processing a **single tweet within a batch**. This helps fully utilize your system’s CPU/GPU resources and speeds up the annotation process.

- **Separation of batch processing:** The actual work for each tweet in a batch is done in the helper function `process_batch`. This function generates responses for all prompts (original, opposite, neutral) for that tweet and returns the results in a structured way.

- **Progress tracking:** We use `tqdm` to show a progress bar for the number of batches processed, giving you visual feedback during long runs.

**Benefits of this approach:**

- Better efficiency and scalability for large datasets.
- Clear separation of batch management and tweet-level processing logic.
- Easier to control batch size and number of worker threads independently.

Overall, this function improves speed and resource management while maintaining the detailed multi-prompt annotation structure from the original function.


In [33]:
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

def generate_scaled_texts(df, model, prompts_with_values, opposite_prompts_with_values, neutral_prompts_with_values, temperature, batch_size=16, num_workers=4):
    """
    Classifies texts using multiple prompts (original, opposite, and neutral) and stores each answer in a separate column.
    This version processes texts in batches.
    """
    # Create column-friendly names for each prompt (original, opposite, and neutral)
    prompt_columns = {prompt_id: f"Prompt_{prompt_id}" for prompt_id in prompts_with_values.keys()}
    opposite_prompt_columns = {f"Opposite_{prompt_id}": f"Opposite_{prompt_id}" for prompt_id in opposite_prompts_with_values.keys()}
    neutral_prompt_columns = {f"Neutral_{prompt_id}": f"Neutral_{prompt_id}" for prompt_id in neutral_prompts_with_values.keys()}

    # Create a dictionary to store results per tweet
    output_data = {}

    # Prepare batch processing
    num_rows = len(df)
    batches = [df.iloc[i:i + batch_size] for i in range(0, num_rows, batch_size)]

    # Use ThreadPoolExecutor with tqdm for progress tracking
    with ThreadPoolExecutor(max_workers=num_workers) as executor:
        for batch in tqdm(batches, desc="Processing Batches", unit="batch"):
            # Process each batch in parallel
            tasks = []
            for _, row in batch.iterrows():
                tasks.append((row, model, temperature))

            # Process each batch in parallel
            batch_results = list(executor.map(process_batch, tasks))

            # Process each row in the batch
            for batch_result in batch_results:
                for tweet_id, result in batch_result.items():
                    if tweet_id not in output_data:
                        output_data[tweet_id] = result

    return pd.DataFrame(output_data.values())  # Convert collected results to DataFrame


def process_batch(task):
    """
    Processes a single batch of rows, extracting the text and generating responses for each prompt.
    """
    row, model, temperature = task
    tweet_id = row['tweet_id']  # Identify the tweet uniquely
    result = {
        **row.to_dict(),  # Preserve original tweet data
        "Model": model,
        "Temperature": temperature
    }

    # Process each original prompt separately
    for prompt_id, prompt in prompts_with_values.items():
        # Original prompt
        response = process_text(row, model, prompt, temperature)
        column_name = f"Prompt_{prompt_id}"
        result[column_name] = response

        # Opposite prompt
        opposite_prompt = opposite_prompts_with_values.get(prompt_id, "")
        if opposite_prompt:  # Ensure opposite prompt exists
            opposite_response = process_text(row, model, opposite_prompt, temperature)
            opposite_column_name = f"Opposite_{prompt_id}"
            result[opposite_column_name] = opposite_response

        # Neutral prompt
        neutral_prompt = neutral_prompts_with_values.get(prompt_id, "")
        if neutral_prompt:  # Ensure neutral prompt exists
            neutral_response = process_text(row, model, neutral_prompt, temperature)
            neutral_column_name = f"Neutral_{prompt_id}"
            result[neutral_column_name] = neutral_response

    return {tweet_id: result}


### Running Batch Annotation on a Subset of Tweets

Here we call the updated `generate_scaled_texts` function to classify a subset of tweets (in this case, `tweets4`, one quarter of the full dataset).

We specify:

- `batch_size=16`: This means the tweets will be processed in small groups of 16 at a time, which helps balance speed and resource use.
- `num_workers=4`: This controls how many threads will run in parallel, allowing multiple batches to be processed simultaneously to speed things up.

The function will return `scaled_results`, a new DataFrame containing the original tweets plus the model’s responses for each prompt in the positive, negative, and neutral categories.

This approach helps manage large datasets efficiently by combining batch processing with multithreading. Please bear in mind that this script was written for the "super computer" which we have at the CEE. The number of workers and batches will definitely have to be adjusted for other machines. If you have high GPU and RAM, you might go up in the number of workers :)


In [35]:
scaled_results = generate_scaled_texts(
    df=tweets4,  # Your DataFrame of tweets (e.g., a subset for testing)
    model=model,  # Pass the model
    prompts_with_values=prompts_with_values, 
    opposite_prompts_with_values=opposite_prompts_with_values, 
    neutral_prompts_with_values=neutral_prompts_with_values,
    temperature=temperature,  # Use a single temperature value
    batch_size=16,  # Number of rows per batch
    num_workers=4  # Number of parallel workers to speed up processing
)

Processing Batches: 100%|██████████| 1141/1141 [15:46:42<00:00, 49.78s/batch]  


### Saving the Annotated Results

After processing the tweets and obtaining the classification results in `scaled_results`, we save the DataFrame to a Parquet file. Parquet is a fast, efficient file format that preserves data types and is well-suited for large datasets. Csv-files get clunky really quickly. parquet is a really good alternative without loosing information.

By saving the results, we can easily reload them later for analysis without having to re-run the annotation process, which can save a lot of time and computing resources. Obviously you will have to change the path to the file on *your* local machine.


In [37]:
# save the results to a parquet file
scaled_results.to_parquet(r'C:\Users\24558\Desktop\chapter-2\data\immigration_classified_tweets_german_parl_scaled_fourth_quarter_mistral.parquet')

In [38]:
# whos entire text
pd.set_option('display.max_colwidth', None)
scaled_results[['text', 'Prompt_1', 'Prompt_2', 'Opposite_1', 'Opposite_2', 'Neutral_1', 'Neutral_2']]

,text,Prompt_1,Prompt_2,Opposite_1,Opposite_2,Neutral_1,Neutral_2
0,@Erbloggtes leider ja. deutsche bevölkerung sind die mit staatsbürgerschaft.,Yes,Yes,Yes,Yes,"No, the tweet does not have a neutral stance as it implies that only those with German citizenship are part of the German population, which could potentially exclude refugees and immigrants who do not yet hold German citizenship.","Yes (The tweet @Erbloggtes unfortunately, yes. The German population is the one with citizenship.)"
1,"merkt mal an: nicht nur die telekommunikation der deutschen bevölkerung sollt sicher sein, sondern die aller hier lebenden menschen","No. The statement does not explicitly express a positive stance on immigration, integration, or refugees. It focuses on the security of communication for all people living here, without specifying who those people are.","Yes. The tweet implies that the communication security of all people living in Germany should be ensured, suggesting a welcoming stance towards those who live there.","No. The provided text does not explicitly express a negative stance on immigration, integration, or refugees. It focuses on the security of communication for all people living here, without mentioning any negative connotations related to immigration or integration.","Yes. The tweet implies that the communication security of only the German population should be ensured, which could potentially suggest a bias or preference towards native Germans over others living in the country. This could be perceived as negative towards immigrants, integration, or refugees.","No. The tweet expresses concern for the security of communication for the German population, but it does not explicitly address immigration, integration, or refugees in a neutral manner.","No. The tweet expresses a concern for the security of communication for all people living here, not just Germans, which could be interpreted as supportive of integration and inclusivity, but it does not explicitly discuss immigration or refugees."
2,"sehr lustig. der ehemalig geheimdienstkoordinator fragt, wann das angefangen hat mit der nsa. er müsste es doch wissen bundestag","No, the tweet does not have a clear positive stance on immigration, integration, or refugees. It appears to be a humorous comment about a former intelligence coordinator asking when something related to the NSA started.","No, the tweet does not have a slightly positive stance on immigration, integration, or refugees. It appears to be a humorous comment about a former intelligence coordinator asking when something related to the NSA started.","No, the tweet does not have a negative stance on immigration, integration, or refugees. It appears to be about the NSA (National Security Agency) and a former intelligence coordinator asking when it started.","No, the tweet does not have a negative stance on immigration, integration, or refugees. It appears to be about the NSA and a former intelligence coordinator asking when it started.","No, the tweet does not have a neutral stance on immigration, integration, or refugees. It appears to be discussing a former intelligence coordinator asking about the start of NSA surveillance, which is unrelated to the topics mentioned.","No, the tweet does not have a neutral stance on immigration, integration, or refugees. It appears to be unrelated to those topics and instead refers to a former intelligence coordinator asking about the start of NSA surveillance."
3,"hat gestrichen &amp; gesäubert, nun wird repräsentiert (kurdistan kultur verein). nur mit kingstons (zwergspitz) freundschaft hapert es noch.",Yes,Yes,"No, the tweet does not have a negative stance on immigration, integration, or refugees. It appears to be about representing Kurdish culture and maintaining friendship with Kingston (a city in Canada), possibly referring to a cultural event or association.",Yes,"No, the tweet does not have a neutral stance on immigration, integration, or refugees as it references a cultural a

In [39]:
# group by party and count the number of opposite_2
scaled_results.groupby('party').count()

,user_username,text,created_at,tweet_id,author_id,source_type,year_month,predicted,Model,Temperature,Prompt_1,Opposite_1,Neutral_1,Prompt_2,Opposite_2,Neutral_2
party,,,,,,,,,,,,,,,,
Linke,18255,18255,18255,18255,18255,18255,18255,18255,18255,18255,18255,18255,18255,18255,18255,18255
